In [1]:
import pandas as pd
import numpy as np
from datetime import datetime
from sklearn.linear_model import LinearRegression
from sklearn.neural_network import MLPRegressor
from lightgbm import LGBMRegressor
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.metrics import mean_squared_error, r2_score
import matplotlib.pyplot as plt
from sklearn.linear_model import Ridge, Lasso

import warnings
warnings.filterwarnings('ignore')

In [2]:
data_ace_24 = pd.read_csv("Ace_погружение_24часа.csv", sep=';', decimal=',', parse_dates=['datetime'], index_col='datetime')
data_discover_24 = pd.read_csv("Discover_погружение_24часа.csv", sep=';', decimal=',', parse_dates=['datetime'], index_col='datetime')
data_ace_af = pd.read_csv("Ace_погружение_AF.csv", sep=';', decimal=',', parse_dates=['datetime'], index_col='datetime')
data_discover_af = pd.read_csv("Discover_погружение_AF.csv", sep=';', decimal=',', parse_dates=['datetime'], index_col='datetime')

In [3]:
def split_datasets_intersection(ace, disc, split_date, scenario):
    """
    Разделяет два датасета ace и discover на train/test по заданной дате split_date.
    При необходимости выравнивает временные индексы (пересечение) и удаляет строки с NaN.

    Сценарии:
    1. 'ace-ace'   — обучение и тестирование на ACE (train до split_date, test после)
    2. 'ace-disc'  — обучение на ACE (до split_date), тестирование на DISC (после split_date)
    3. 'disc-ace'  — обучение на DISC (до split_date), тестирование на ACE (после split_date)
    4. 'disc-disc' — обучение и тестирование на DISC (train до split_date, test после)
    """
    a = ace.copy()
    d = disc.copy()

    # Выравниваем временные индексы так, чтобы все наборы были на одинаковых данных в пересечениях
    common_idx = a.index.intersection(d.index)
    a = a.loc[common_idx].dropna(axis=0, how='any')
    d = d.loc[common_idx].dropna(axis=0, how='any')

    # Дата разделения
    split_date = pd.Timestamp(split_date)

    # Сценарии
    if scenario == 'ace-ace':
        train = a[a.index < split_date].copy()
        test  = a[a.index >= split_date].copy()

    elif scenario == 'ace-disc':
        train = a[a.index < split_date].copy()
        test  = d[d.index >= split_date].copy()

    elif scenario == 'disc-ace':
        train = d[d.index < split_date].copy()
        test  = a[a.index >= split_date].copy()

    elif scenario == 'disc-disc':
        train = d[d.index < split_date].copy()
        test  = d[d.index >= split_date].copy()

    else:
        raise ValueError(f"Некорректный сценарий '{scenario}'. "
                         f"Должен быть один из: 'ace-ace', 'ace-disc', 'disc-ace', 'disc-disc'")

    return train, test

In [45]:
def models_shuffle(ace, disc, split_date, all_targets, target_col):
    """
    Объединяет данные ace и discover, добавляет признак is_ace,
    разделяет на train/test по дате и обучает модели.
    """
    ace['is_ace'] = 1
    disc['is_ace'] = 0

    split_date = pd.Timestamp(split_date)
    
    combined_data = pd.concat([ace, disc], ignore_index=False)
    combined_data = combined_data.sort_values('datetime')
    
    train = combined_data[combined_data.index < split_date].copy()
    test = combined_data[combined_data.index >= split_date].copy()
    
    feature_cols = [c for c in train.columns if c not in all_targets and c != 'date']
    
    train = train.dropna(subset=feature_cols + [target_col])
    test = test.dropna(subset=feature_cols + [target_col])

    X_train, y_train = train[feature_cols].values, train[target_col].values
    X_test, y_test = test[feature_cols].values, test[target_col].values
    
    results = {}

    for name, model in models.items():
        try:
            model.fit(X_train, y_train)
            y_pred = model.predict(X_test)
            mse = mean_squared_error(y_test, y_pred)
            r2 = r2_score(y_test, y_pred)
            results[name] = {'mse': mse, 'r2': r2}
            print(f"{name:10s} -> mse: {mse:.4f}, r2: {r2:.4f}")
        except Exception as e:
            print(f"{name:10s} -> Ошибка обучения: {e}")
            results[name] = {'mse': None, 'r2': None}

    return results

In [4]:
def build_models(random_state=42):
    models = {}
    models['Linear'] = Pipeline([
        ("scaler", StandardScaler()),
        ("lin", LinearRegression())
    ])
    
    models['Ridge'] = Pipeline([
        ("scaler", StandardScaler()),
        ("ridge", Ridge(
            alpha=1.0,
            solver='auto',
            random_state=random_state))
    ])

    models['Lasso'] = Pipeline([
        ("scaler", StandardScaler()),
        ("lasso", Lasso(
            alpha=0.0005,
            tol=0.01,
            max_iter=500, 
            random_state=random_state))
    ])
    
    models['LGBM'] = Pipeline([
        ("scaler", StandardScaler()),
        ("boost", LGBMRegressor(
        metric='mse',
        n_estimators=1500,
        learning_rate=0.03,
        max_depth=-1,
        num_leaves=64,
        subsample=0.8,
        colsample_bytree=0.8,
        reg_lambda=0.1,
        reg_alpha=0.05,
        random_state=random_state,
        n_jobs=-1,
        verbose=-1))
    ])
    
    models['MLP'] = Pipeline([
        ("scaler", StandardScaler()),
        ("mlp", MLPRegressor(
            hidden_layer_sizes=(256, 128, 64),
            activation='relu',
            alpha=0.0005,
            learning_rate_init=0.001,
            solver='adam',
            early_stopping=True,
            validation_fraction=0.1,
            max_iter=1000,
            random_state=random_state))
    ])
    return models
    
models = build_models()

def evaluate_models(ace, disc, split_date, all_targets, target_col, scenario='ace-ace'):
    """
    Обучает все модели на train/test наборах, полученных из split_datasets_intersection(),
    и вычисляет метрики mse и r2_score

    Выход:
    results : dict
        Метрики по каждой модели: {model_name: {'mse': ..., 'r2': ...}}
    """
    # Разделяем данные по сценарию
    train, test = split_datasets_intersection(ace, disc, split_date, scenario=scenario)

    # Проверка: если пусто — пропускаем
    if len(train) == 0 or len(test) == 0:
        print(f"[{scenario}] Ошибка - пустые train/test после разделения")
        return None

    # Определение признаков по заданным таргетам (здесь убираю из признаков все значения dst+1, dst+2, dst+3)
    feature_cols = [c for c in train.columns if c not in all_targets]
    
    train = train.dropna(subset=feature_cols + [target_col])
    test = test.dropna(subset=feature_cols + [target_col])

    X_train, y_train = train[feature_cols].values, train[target_col].values
    X_test, y_test = test[feature_cols].values, test[target_col].values
    
    results = {}

    print(f"\n=== {scenario.upper()} ===")
    for name, model in models.items():
        try:
            model.fit(X_train, y_train)
            y_pred = model.predict(X_test)
            mse = mean_squared_error(y_test, y_pred)
            r2 = r2_score(y_test, y_pred)
            results[name] = {'mse': mse, 'r2': r2}
            print(f"{name:10s} -> mse: {mse:.4f}, r2: {r2:.4f}")
        except Exception as e:
            print(f"{name:10s} -> Ошибка обучения: {e}")
            results[name] = {'mse': None, 'r2': None}

    return results

In [5]:
# 0. Копирование загруженных данных в отдельные переменные
data_ace_24_copy = data_ace_24.copy()
data_discover_24_copy = data_discover_24.copy()
data_ace_af_copy = data_ace_af.copy()
data_discover_af_copy = data_discover_af.copy()

# 1. Задание переменных для обучения
split_date = "2021-01-01" # дата по которой происходит разбиение данных на тренировочный и тестовый наборы
scenarios = ['ace-ace', 'ace-disc', 'disc-ace', 'disc-disc']
targets = ['Dst_plus1', 'Dst_plus2', 'Dst_plus3']
# targets = ['Dst_plus2']
results_all = {}

In [6]:
# 2. Обучение для датасетов с глубиной - 24 часа по всем переменным
print(f"\n==== Depth - 24h ====")
for target_col in targets:
    print(f"\n==== Forecast of {target_col.upper()} ====")
    for scen in scenarios:
        res = evaluate_models(data_ace_24_copy, data_discover_24_copy, split_date, targets, target_col, scenario=scen)
        results_all[scen] = res


==== Depth - 24h ====

==== Forecast of DST_PLUS1 ====

=== ACE-ACE ===
Linear     -> mse: 10.7099, r2: 0.9635
Ridge      -> mse: 10.7116, r2: 0.9635
Lasso      -> mse: 10.7140, r2: 0.9635
LGBM       -> mse: 14.8083, r2: 0.9496
MLP        -> mse: 13.0467, r2: 0.9556

=== ACE-DISC ===
Linear     -> mse: 11.6129, r2: 0.9604
Ridge      -> mse: 11.6106, r2: 0.9605
Lasso      -> mse: 11.4988, r2: 0.9608
LGBM       -> mse: 15.0017, r2: 0.9489
MLP        -> mse: 17.8781, r2: 0.9391

=== DISC-ACE ===
Linear     -> mse: 10.7695, r2: 0.9633
Ridge      -> mse: 10.7710, r2: 0.9633
Lasso      -> mse: 10.7733, r2: 0.9633
LGBM       -> mse: 15.5340, r2: 0.9471
MLP        -> mse: 13.0361, r2: 0.9556

=== DISC-DISC ===
Linear     -> mse: 10.4346, r2: 0.9645
Ridge      -> mse: 10.4354, r2: 0.9645
Lasso      -> mse: 10.4342, r2: 0.9645
LGBM       -> mse: 14.5516, r2: 0.9504
MLP        -> mse: 12.9958, r2: 0.9557

==== Forecast of DST_PLUS2 ====

=== ACE-ACE ===
Linear     -> mse: 26.5068, r2: 0.9097
Rid

In [7]:
# 3. Обучение для датасетов с глубиной, заданной автокорреляционной функцией
print(f"\n==== Depth - autocorrelation function ====")
for target_col in targets:
    print(f"\n==== Forecast of {target_col.upper()} ====")
    for scen in scenarios:
        res = evaluate_models(data_ace_af_copy, data_discover_af_copy, split_date, targets, target_col, scenario=scen)
        results_all[scen] = res


==== Depth - autocorrelation function ====

==== Forecast of DST_PLUS1 ====

=== ACE-ACE ===
Linear     -> mse: 10.7296, r2: 0.9633
Ridge      -> mse: 10.7310, r2: 0.9633
Lasso      -> mse: 10.7276, r2: 0.9633
LGBM       -> mse: 14.4501, r2: 0.9506
MLP        -> mse: 12.0669, r2: 0.9588

=== ACE-DISC ===
Linear     -> mse: 11.6255, r2: 0.9603
Ridge      -> mse: 11.6226, r2: 0.9603
Lasso      -> mse: 11.5065, r2: 0.9607
LGBM       -> mse: 14.8220, r2: 0.9493
MLP        -> mse: 16.4927, r2: 0.9436

=== DISC-ACE ===
Linear     -> mse: 10.8149, r2: 0.9630
Ridge      -> mse: 10.8163, r2: 0.9630
Lasso      -> mse: 10.8199, r2: 0.9630
LGBM       -> mse: 15.4139, r2: 0.9473
MLP        -> mse: 12.0118, r2: 0.9589

=== DISC-DISC ===
Linear     -> mse: 10.4570, r2: 0.9643
Ridge      -> mse: 10.4576, r2: 0.9643
Lasso      -> mse: 10.4521, r2: 0.9643
LGBM       -> mse: 14.5755, r2: 0.9502
MLP        -> mse: 12.5005, r2: 0.9573

==== Forecast of DST_PLUS2 ====

=== ACE-ACE ===
Linear     -> mse: 26

In [47]:
# 4. Предсказание на перемешанных данных с признаком is_ace
print(f"\n==== Depth - 24h ====")
for target_col in targets:
    print(f"\n==== Forecast of {target_col.upper()} ====")
    res = models_shuffle(data_ace_24_copy, data_discover_24_copy, split_date, targets, target_col)
    results_all[scen] = res

print(f"\n==== Depth - autocorrelation function ====")
for target_col in targets:
    print(f"\n==== Forecast of {target_col.upper()} ====")
    res = models_shuffle(data_ace_af_copy, data_discover_af_copy, split_date, targets, target_col)
    results_all[scen] = res


==== Depth - 24h ====

==== Forecast of DST_PLUS1 ====
Linear     -> mse: 10.4424, r2: 0.9643
Ridge      -> mse: 10.4423, r2: 0.9643
Lasso      -> mse: 10.4327, r2: 0.9644
LGBM       -> mse: 10.5789, r2: 0.9639
MLP        -> mse: 10.2891, r2: 0.9649

==== Forecast of DST_PLUS2 ====
Linear     -> mse: 25.4837, r2: 0.9130
Ridge      -> mse: 25.4835, r2: 0.9130
Lasso      -> mse: 25.4624, r2: 0.9130
LGBM       -> mse: 23.0214, r2: 0.9214
MLP        -> mse: 25.5010, r2: 0.9129

==== Forecast of DST_PLUS3 ====
Linear     -> mse: 41.5241, r2: 0.8581
Ridge      -> mse: 41.5240, r2: 0.8581
Lasso      -> mse: 41.5113, r2: 0.8582
LGBM       -> mse: 37.6273, r2: 0.8715
MLP        -> mse: 50.7905, r2: 0.8265

==== Depth - autocorrelation function ====

==== Forecast of DST_PLUS1 ====
Linear     -> mse: 10.5125, r2: 0.9640
Ridge      -> mse: 10.5124, r2: 0.9640
Lasso      -> mse: 10.5022, r2: 0.9641
LGBM       -> mse: 10.8425, r2: 0.9629
MLP        -> mse: 10.4213, r2: 0.9643

==== Forecast of DST